In [33]:
import pandas as pd

In [34]:
test_df = pd.read_csv("./inputs/test.csv").set_index("id")
test_df

,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment
id,,,,,,,,,,
750000,Educational Nuggets,Episode 73,78.96,Education,38.11,Saturday,Evening,53.33,1.0,Neutral
750001,Sound Waves,Episode 23,27.87,Music,71.29,Sunday,Morning,NaN,0.0,Neutral
750002,Joke Junction,Episode 11,69.10,Comedy,67.89,Friday,Evening,97.51,0.0,Positive
750003,Comedy Corner,Episode 73,115.39,Comedy,23.40,Sunday,Morning,51.75,2.0,Positive
750004,Life Lessons,Episode 50,72.32,Lifestyle,58.10,Wednesday,Morning,11.30,2.0,Neutral
...,...,...,...,...,...,...,...,...,...,...
999995,Mind & Body,Episode 100,21.05,Health,65.77,Saturday,Evening,96.40,3.0,Negative
999996,Joke Junction,Episode 85,85.50,Comedy,41.47,Saturday,Night,30.52,2.0,Negative
999997,Joke Junction,Episode 63,12.11,Comedy,25.92,Thursday,Evening,73.69,1.0,Neutral


In [35]:
%store -r categories
%store -r non_categories

In [36]:
# Missing Data?
test_df.isnull().sum()

Podcast_Name                       0
Episode_Title                      0
Episode_Length_minutes         28736
Genre                              0
Host_Popularity_percentage         0
Publication_Day                    0
Publication_Time                   0
Guest_Popularity_percentage    48832
Number_of_Ads                      0
Episode_Sentiment                  0
dtype: int64

In [37]:
# Create a copy of test_df
test_df_copy = test_df.copy()

In [38]:
# First, split the dataset into missing values for ELm (Episode_Listening_minutes) and non-missing data
#
# Data that does not have any missing values
non_missing = test_df_copy.dropna(subset=["Episode_Length_minutes","Guest_Popularity_percentage"])

# Data that has all missing ELm values
missing_ELm = test_df_copy[test_df_copy["Episode_Length_minutes"].isnull()]

# Data that has all missing GPl values
missing_GPl = test_df_copy[test_df_copy["Guest_Popularity_percentage"].isnull()]

In [39]:
# Encode the categorical features
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for column in categories:
    non_missing.loc[:, column] = le.fit_transform(non_missing[column])
    missing_ELm.loc[:, column] = le.fit_transform(missing_ELm[column])
    missing_GPl.loc[:, column] = le.fit_transform(missing_GPl[column])

non_missing

,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment
id,,,,,,,,,,
750000,11,71,78.96,2,38.11,2,1,53.33,1.0,1
750002,24,3,69.10,1,67.89,0,1,97.51,0.0,2
750003,4,71,115.39,1,23.40,3,2,51.75,2.0,2
750004,27,46,72.32,4,58.10,6,2,11.30,2.0,1
750006,34,27,116.09,9,27.57,0,3,22.82,1.0,2
...,...,...,...,...,...,...,...,...,...,...
999995,31,2,21.05,3,65.77,2,1,96.40,3.0,0
999996,24,84,85.50,1,41.47,2,3,30.52,2.0,0
999997,24,60,12.11,1,25.92,4,1,73.69,1.0,1


In [40]:
# Encode train_df_copy categorical features
for column in categories:
    test_df_copy[column] = le.fit_transform(test_df_copy[column])

In [41]:
# Splitting the data once again
# Data that has all missing ELm values and also some missing GPl values
missing_elm = missing_ELm[missing_ELm["Guest_Popularity_percentage"].notnull()]

# Data that has both missing ELm and GPl values from missing_ELm
missing_both_elm = missing_ELm[missing_ELm["Guest_Popularity_percentage"].isnull()]

# Data that has all missing GPl values and also some missing ELm values
missing_gpl = missing_GPl[missing_GPl["Episode_Length_minutes"].notnull()]

# Data that has both missing GPl and ELm values from missing_GPl
missing_both_gpl = missing_GPl[missing_GPl["Episode_Length_minutes"].isnull()]

In [42]:
# Predictive imputing on ELm
from sklearn.ensemble import RandomForestRegressor

# Step 1: Train model on known data
X_elm_train = non_missing.drop(columns=["Episode_Length_minutes"])
y_elm_train = non_missing["Episode_Length_minutes"]

elm_model = RandomForestRegressor()
elm_model.fit(X_elm_train, y_elm_train)

# Step 2: Predict missing ELm where GPl is available
X_elm_predict = missing_elm.drop(columns=["Episode_Length_minutes"])
elm_preds = elm_model.predict(X_elm_predict)

# Step 3: Fill predictions
test_df.loc[missing_elm.index, "Episode_Length_minutes"] = elm_preds

In [43]:
# Predictive imputing on GPl
# Step 1: Update non_missing to include rows with newly filled ELm
non_missing_gpl = test_df_copy.dropna(subset=["Guest_Popularity_percentage", "Episode_Length_minutes"])

# Step 2: Train model to predict GPl
X_gpl_train = non_missing_gpl.drop(columns=["Guest_Popularity_percentage"])
y_gpl_train = non_missing_gpl["Guest_Popularity_percentage"]

gpl_model = RandomForestRegressor()
gpl_model.fit(X_gpl_train, y_gpl_train)

# Step 3: Predict GPl for rows with missing GPl and known ELm
X_gpl_predict = missing_gpl.drop(columns=["Guest_Popularity_percentage"])
gpl_preds = gpl_model.predict(X_gpl_predict)

# Step 4: Fill predictions
test_df.loc[missing_gpl.index, "Guest_Popularity_percentage"] = gpl_preds

In [44]:
# Handling missing_both_elm and missing_both_gpl
# Fill Episode_Length_minutes in missing_both_elm
X_both_elm = missing_both_elm.drop(columns=["Episode_Length_minutes"])
elm_preds_both = elm_model.predict(X_both_elm)
test_df.loc[missing_both_elm.index, "Episode_Length_minutes"] = elm_preds_both

# Fill Guest_Popularity_percentage in missing_both_gpl
X_both_gpl = missing_both_gpl.drop(columns=["Guest_Popularity_percentage"])
gpl_preds_both = gpl_model.predict(X_both_gpl)
test_df.loc[missing_both_gpl.index, "Guest_Popularity_percentage"] = gpl_preds_both

In [ ]:
# Missing Values?
test_df.isnull().sum()

In [45]:
test_df

,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment
id,,,,,,,,,,
750000,Educational Nuggets,Episode 73,78.96,Education,38.11,Saturday,Evening,53.3300,1.0,Neutral
750001,Sound Waves,Episode 23,27.87,Music,71.29,Sunday,Morning,42.6747,0.0,Neutral
750002,Joke Junction,Episode 11,69.10,Comedy,67.89,Friday,Evening,97.5100,0.0,Positive
750003,Comedy Corner,Episode 73,115.39,Comedy,23.40,Sunday,Morning,51.7500,2.0,Positive
750004,Life Lessons,Episode 50,72.32,Lifestyle,58.10,Wednesday,Morning,11.3000,2.0,Neutral
...,...,...,...,...,...,...,...,...,...,...
999995,Mind & Body,Episode 100,21.05,Health,65.77,Saturday,Evening,96.4000,3.0,Negative
999996,Joke Junction,Episode 85,85.50,Comedy,41.47,Saturday,Night,30.5200,2.0,Negative
999997,Joke Junction,Episode 63,12.11,Comedy,25.92,Thursday,Evening,73.6900,1.0,Neutral


In [47]:
%store test_df

Stored 'test_df' (DataFrame)
